# Test Accuracy: Baseline vs N2UQ vs TLMAC

Compares three inference paths on identical inputs:
1. **Baseline** — standard torchvision ResNet-18 (float32 weights)
2. **N2UQ** — Nonuniform-to-Uniform Quantised ResNet-18 (3-bit, from `.pth.tar`)
3. **TLMAC** — emulated TLMAC PE: loads LUT INIT hex files, performs
   bit-serial LUT lookup convolution exactly as the FPGA PE would
   (paper §3.1.2, §4). Non-conv layers reuse N2UQ parameters.

**Priority**: accurate evaluation of TLMAC architecture and compiled
weights versus N2UQ. TLMAC weights are derived from N2UQ via clustering
and routing SA (see `tlmac_compile.ipynb`), so we compare **conv outputs**
and **end-to-end accuracy**, not bit-exact weights.

Section 3 verifies LUT INIT hex files decode correctly (emulator integrity).
Section 4 is the primary per-layer accuracy test (same activations in,
compare outputs out). Section 5 is the full-model end-to-end comparison.


In [ ]:
!apt-get install -y git-lfs > /dev/null 2>&1 && git lfs install
!git clone https://github.com/leozqi/resnet18_n2uq.git || true
%cd resnet18_n2uq
!pip install torch torchvision numpy matplotlib 2>/dev/null

# Unpack TLMAC hex files if present
import os, tarfile
for tarname in ['tlmac_output_new.tar.gz', 'tlmac_output.tar.gz']:
    if os.path.exists(tarname):
        with tarfile.open(tarname, 'r:gz') as tar:
            tar.extractall('.')
        print(f'Extracted {tarname}')
        break


In [ ]:
import math, os, json, random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision.models as vision_models

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 1. Model Definitions


In [ ]:
# --- N2UQ architecture (from resnet.py) ---
N_BIT = 3

class LearnableBias(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, ch, 1, 1))
    def forward(self, x):
        return x + self.bias.expand_as(x)

class LTQ(nn.Module):
    def __init__(self, num_bits):
        super().__init__()
        self.n_val = 2 ** num_bits - 1
        self.interval = 2.0 / self.n_val
        self.start = nn.Parameter(torch.tensor([0.0]))
        self.a = nn.Parameter(torch.tensor([self.interval] * self.n_val))
        self.scale1 = nn.Parameter(torch.tensor([1.0]))
        self.scale2 = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x):
        x = x * self.scale1
        a_pos = torch.where(self.a > 1e-3, self.a, torch.tensor(1e-3))
        step_right = torch.tensor(0.0)
        x_f = x; x_b = x
        for i in range(self.n_val):
            step_right = step_right + self.interval
            if i == 0:
                th_f = self.start + a_pos[0] / 2
                th_b = self.start
                x_f = torch.where(x > th_f, step_right, torch.tensor(0.0))
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, torch.tensor(0.0))
            else:
                th_f = th_f + a_pos[i - 1] / 2 + a_pos[i] / 2
                th_b = th_b + a_pos[i - 1]
                x_f = torch.where(x > th_f, step_right, x_f)
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, x_b)
        th_b = th_b + a_pos[-1]
        x_b = torch.where(x > th_b, torch.tensor(2.0), x_b)
        return (x_f.detach() + x_b - x_b.detach()) * self.scale2

class HardQuantizeConv(nn.Module):
    def __init__(self, in_ch, out_ch, num_bits, ks=3, stride=1, padding=1):
        super().__init__()
        self.stride = stride; self.padding = padding
        self.num_bits = num_bits; self.kernel_size = ks
        self.clip_val = nn.Parameter(torch.tensor([2.0]))
        self.shape = (out_ch, in_ch, ks, ks)
        self.weight = nn.Parameter((torch.rand(self.shape) - 0.5) * 0.001)

    def quantise_fwd(self):
        rw = self.weight
        gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
        sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True).detach()
        sw = rw / sf
        cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
        n = float(2 ** self.num_bits - 1) / self.clip_val
        return sf * (torch.round((cw + self.clip_val / 2) * n) / n - self.clip_val / 2)

    def forward(self, x):
        return F.conv2d(x, self.quantise_fwd(), stride=self.stride, padding=self.padding)

    def get_integer_weights(self):
        with torch.no_grad():
            rw = self.weight
            gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
            sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True)
            sw = rw / sf
            cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
            n = float(2 ** self.num_bits - 1) / self.clip_val
            return torch.round((cw + self.clip_val / 2) * n).to(torch.int8)  # 0..2^n-1

    def get_quantised_real_weights(self):
        return self.quantise_fwd().detach()

    def get_sf(self):
        """Per-output-channel scaling factor sf = gamma * mean(|w|)."""
        rw = self.weight
        gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
        return gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True).detach()


In [ ]:
class BasicBlockQ(nn.Module):
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bias11 = LearnableBias(inplanes)
        self.prelu1 = nn.PReLU(inplanes)
        self.bias12 = LearnableBias(inplanes)
        self.quan1 = LTQ(N_BIT)
        self.conv1 = HardQuantizeConv(inplanes, planes, N_BIT, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.bias21 = LearnableBias(planes)
        self.prelu2 = nn.PReLU(planes)
        self.bias22 = LearnableBias(planes)
        self.quan2 = LTQ(N_BIT)
        self.conv2 = HardQuantizeConv(planes, planes, N_BIT, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.bias31 = LearnableBias(planes)
        self.prelu3 = nn.PReLU(planes)
        self.bias32 = LearnableBias(planes)
    def forward(self, x):
        identity = x
        out = self.bias12(self.prelu1(self.bias11(x)))
        out = self.bn1(self.conv1(self.quan1(out)))
        out = self.bias22(self.prelu2(self.bias21(out)))
        out = self.bn2(self.conv2(self.quan2(out)))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return self.bias32(self.prelu3(self.bias31(out)))

class N2UQ_ResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._blk(64, 64, 2)
        self.layer2 = self._blk(64, 128, 2, stride=2)
        self.layer3 = self._blk(128, 256, 2, stride=2)
        self.layer4 = self._blk(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, 1000)
    def _blk(self, inp, planes, blocks, stride=1):
        ds = None
        if stride != 1 or inp != planes:
            ds = nn.Sequential(LTQ(N_BIT), HardQuantizeConv(inp, planes, N_BIT, 1, stride, 0), nn.BatchNorm2d(planes))
        layers = [BasicBlockQ(inp, planes, stride, ds)]
        for _ in range(1, blocks): layers.append(BasicBlockQ(planes, planes))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1))


### Load Models


In [ ]:
# --- N2UQ model ---
model_n2uq = N2UQ_ResNet18().to(device)
n2uq_ckpt = torch.load('pretrained_models/quantize_downsampling_true/real_res18.pth.tar', map_location='cpu')
if isinstance(n2uq_ckpt, dict) and 'state_dict' in n2uq_ckpt:
    n2uq_ckpt = n2uq_ckpt['state_dict']
model_n2uq.load_state_dict(n2uq_ckpt, strict=False)
model_n2uq.eval()
for p in model_n2uq.parameters(): p.requires_grad_(False)

# --- Baseline model ---
model_bl = vision_models.resnet18(weights=None)
model_bl.load_state_dict(torch.load('pretrained_models/quantize_downsampling_false/resnet18-f37072fd(pytorch_model).pth', map_location='cpu'))
model_bl = model_bl.to(device).eval()
for p in model_bl.parameters(): p.requires_grad_(False)

print(f'N2UQ ResNet-18:     {sum(p.numel() for p in model_n2uq.parameters()):,} params')
print(f'Baseline ResNet-18: {sum(p.numel() for p in model_bl.parameters()):,} params')


In [ ]:
x_test = torch.randn(4, 3, 224, 224, device=device)
with torch.no_grad():
    out_bl = model_bl(x_test)
    out_n2uq = model_n2uq(x_test)
print(f'Baseline shape: {out_bl.shape}, top-1: {out_bl.argmax(1).tolist()}')
print(f'N2UQ shape:     {out_n2uq.shape}, top-1: {out_n2uq.argmax(1).tolist()}')


### Load Test Images

Use torchvision's built-in sample images and standard ImageNet
preprocessing (resize, center crop, normalise) to produce realistic
inputs that exercise the learned LTQ thresholds.


In [ ]:
from torchvision import transforms
from PIL import Image

# ImageNet preprocessing (same as torchvision pretrained models)
imagenet_preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# torchvision ships a few sample images; dog2.jpg is the canonical
# ResNet test image used in PyTorch tutorials.
import torchvision as _tv
sample_dir = os.path.join(os.path.dirname(_tv.__file__), 'samples')

test_images = []
if os.path.isdir(sample_dir):
    for fname in sorted(os.listdir(sample_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img = Image.open(os.path.join(sample_dir, fname)).convert('RGB')
            test_images.append(imagenet_preprocess(img))
    print(f'Loaded {len(test_images)} torchvision sample images from {sample_dir}')

# Fallback: download the iconic ResNet test image (golden retriever)
if len(test_images) == 0:
    import urllib.request
    url = ('https://github.com/pytorch/hub/raw/master/images/dog.jpg')
    fname = '/tmp/dog.jpg'
    try:
        urllib.request.urlretrieve(url, fname)
        img = Image.open(fname).convert('RGB')
        test_images.append(imagenet_preprocess(img))
        print('Downloaded dog.jpg (ImageNet class 207: golden retriever)')
    except Exception as e:
        print(f'Download failed: {e}')

x_test_images = torch.stack(test_images).to(device)
N_IMG = x_test_images.shape[0]
print(f'Test images: {N_IMG}, shape: {x_test_images.shape}')


## 2. TLMAC Software Emulator

Faithfully reproduces the TLMAC PE hardware operations from the paper
(§3.1.2 Hybrid Bit-Serial, §4 Hardware Architecture) for 3-bit weights
and 3-bit activations.

### Paper §3.1.2 — Bit-Serial MAC

$$p = \sum_{b=0}^{B_a-1} 2^b \left( a_0^b \cdot w_0 + a_1^b \cdot w_1 + \ldots + a_{G-1}^b \cdot w_{G-1} \right)$$

For 3-bit weights ($B_w=3$) and 3-bit activations ($B_a=3$), with $G=3$
(one kernel row):
- Each LUT array stores $N_{clus}=2^{6-G}=8$ weight groups selectable
  by a 3-bit select signal (the `step`→`wg_sel` mapping).
- $N_{lut} = B_w + \lceil\log_2 G\rceil = 3+2 = 5$ LUT-6 per array.
- Bit-serial: 3 iterations (LSB first), shift-left by $b$ and accumulate.

### Paper §4 — Hardware Flow

1. Load partial sums from buffer into $D_p$ accumulators.
2. For each step $s \in [0, D_s-1]$ (constant `wg_sel` from cluster_map ROM):
   - For each bit-serial iteration $b = 0, 1, 2$ (LSB→MSB):
     - Feed activation bits $a_g^b$ to all LUT arrays.
     - LUT array computes $\sum_g a_g^b \cdot w_g$ (signed integer MAC).
     - Switch network routes array output to the correct accumulator.
     - Shift left by $b$, add to accumulator.
3. After all $B_a$ bits, output is the integer partial sum.

### Activation Quantization

The N2UQ LTQ produces activations in $[0, 2]$ with 3-bit resolution.
The integer activation level is $a_g = \text{round}(x_g \cdot \frac{2^{B_a}-1}{2})$,
giving values $\{0, 1, \ldots, 7\}$.

### Scaling: TLMAC psum → N2UQ float output

The N2UQ conv computes $y = F.\text{conv2d}(x, q_w)$ where
$q_w = s_f \cdot (q_i/n - \text{clip}/2)$ and $w_{int} = q_i - W_{offset}$.
Since $W_{offset}/n = \text{clip}/2$, this simplifies to
$y = \frac{s_f}{n} \sum_g x_g \cdot w_{int,g}$.

The TLMAC psum $= \sum_g a_g \cdot w_{int,g}$ where $a_g \approx n_q \cdot x_g$,
so $y = \frac{s_f}{n \cdot n_q} \cdot \text{psum}$ where $n_q = \frac{2^{B_a}-1}{2}$.


In [ ]:
class TLMACConvEmulator:
    """Bit-serial TLMAC PE emulator for 3-bit weights, 3-bit activations.

    Loads hex files from tlmac_output/ and performs the exact LUT-based
    convolution the FPGA PE would execute (paper §3.1.2, §4):
      - bit-serial over activation bits (LSB first)
      - LUT-6 lookup with weight-group select
      - switch-network routing to accumulators
      - shift-left accumulate

    Parameters (from metadata.json):
      G=3, BW=3, BA=3, NCLUS=8, NLUT_PER_ARR=5, WEIGHT_OFFSET=3
    """

    def __init__(self, layer_dir, module, meta):
        self.module = module
        D_o, D_i, Dk, _ = module.shape
        self.D_o = D_o
        self.D_i = D_i
        self.DK = Dk
        s = module.stride
        self.stride = s if isinstance(s, int) else s[0]
        p = module.padding
        self.padding = p if isinstance(p, int) else p[0]
        self.D_s = meta['D_s']
        self.D_p = meta['D_p']  # 64 * DK = 192
        self.n_arr = meta['n_arr']
        self.NLUT = meta['NLUT_PER_ARR']  # 5
        self.NCLUS = meta['NCLUS']       # 8
        self.G = meta['G']               # 3
        self.BW = meta['BW']             # 3
        self.BA = meta.get('BA', meta['BW'])  # 3
        self.WEIGHT_OFFSET = meta.get('WEIGHT_OFFSET', (2**self.BW - 1) // 2)  # 3
        self.o_blocks = D_o // 64

        # --- cluster_map.hex: step -> wg_sel (3-bit select) ---
        self.cluster_map = []
        with open(os.path.join(layer_dir, 'cluster_map.hex')) as f:
            for line in f:
                self.cluster_map.append(int(line.strip(), 16))

        # --- LUT INIT files: lut_arr{ai}.hex ---
        # Each file: NLUT_PER_ARR lines of 64-bit hex.
        # LUT-6 address = {wg_sel[NCLUS_bits-1:0], act_bits[G-1:0]} = 6 bits.
        # INIT bit[addr] = output bit k for that address.
        self.lut_init = torch.zeros(self.n_arr, self.NLUT, dtype=torch.int64)
        for ai in range(self.n_arr):
            with open(os.path.join(layer_dir, f'lut_arr{ai}.hex')) as f:
                for k, line in enumerate(f):
                    self.lut_init[ai, k] = int(line.strip(), 16)

        # Precompute lut_table[ai, addr] = NLUT-bit signed MAC result.
        # addr = 6 bits = {wg_sel[2:0], act_bits[2:0]}
        addr_range = torch.arange(64)
        table = torch.zeros(self.n_arr, 64, dtype=torch.int64)
        for k in range(self.NLUT):
            table |= ((self.lut_init[:, k].unsqueeze(1) >> addr_range) & 1) << k
        # Sign-extend: NLUT=5 bits, range [-16, +15]
        sign_bit = 1 << (self.NLUT - 1)
        self.lut_table = torch.where(
            table >= sign_bit, table - (1 << self.NLUT), table
        )  # [n_arr, 64] signed int64

        # --- route_map.hex: per-step routing [D_s, D_p] ---
        # Paper §4: per-step routing ROM. route_map[s, p] = array
        # driving output p at step s.
        self.route_map = torch.zeros(self.D_s, self.D_p, dtype=torch.int64)
        with open(os.path.join(layer_dir, 'route_map.hex')) as f:
            for s in range(self.D_s):
                for p in range(self.D_p):
                    self.route_map[s, p] = int(f.readline().strip(), 16)

        # Precompute routed_table[s, p, addr] = lut_table[route_map[s,p], addr]
        # This is the per-step MUX selection from the routing ROM (paper §4).
        self.routed_table = self.lut_table[self.route_map]  # [D_s, D_p, 64]

        # Precompute spatial index helpers
        # For each step s: ic = s // o_blocks, o_blk = s % o_blocks
        s_range = torch.arange(self.D_s)
        self.step_ic = (s_range // self.o_blocks).to(torch.int64)
        self.step_oblk = (s_range % self.o_blocks).to(torch.int64)
        self.step_wgsel = torch.tensor(self.cluster_map, dtype=torch.int64)

    def forward(self, x):
        """TLMAC bit-serial convolution (paper §3.1.2, §4).

        x: [batch, D_i, H, W] float activations (LTQ output, range [0, 2]).
        Returns: [batch, D_o, oH, oW] integer partial sums (float32 tensor).
        """
        B, _, H, W = x.shape
        stride, pad, DK = self.stride, self.padding, self.DK
        oH = (H + 2 * pad - DK) // stride + 1
        oW = (W + 2 * pad - DK) // stride + 1
        x_pad = F.pad(x, [pad, pad, pad, pad])

        # Quantize activations: LTQ output [0, 2] -> BA-bit unsigned [0, 2^BA-1]
        n_q = (2 ** self.BA - 1) / 2.0  # 3.5 for BA=3
        x_q = torch.round(x_pad * n_q).clamp(0, 2 ** self.BA - 1).to(torch.int64)

        # Spatial index grids (vectorised over all oH*oW positions)
        oh_idx = torch.arange(oH).unsqueeze(1).expand(-1, oW).reshape(-1)  # [N_spatial]
        ow_idx = torch.arange(oW).unsqueeze(0).expand(oH, -1).reshape(-1)
        N_spatial = oH * oW
        w_offsets = torch.arange(DK)  # [DK]

        output = torch.zeros(B, self.D_o, oH, oW, dtype=torch.float32,
                             device=x.device)

        for b_idx in range(B):
            psum = torch.zeros(N_spatial, self.D_o, dtype=torch.int64,
                                device=x.device)

            for s in range(self.D_s):
                ic = int(self.step_ic[s].item())
                o_blk = int(self.step_oblk[s].item())
                wg_sel = int(self.step_wgsel[s].item())

                for kr in range(DK):
                    # Gather activation values for all spatial positions
                    h_pos = oh_idx * stride + kr          # [N_spatial]
                    w_start = ow_idx * stride              # [N_spatial]
                    w_idx = (w_start.unsqueeze(1) + w_offsets.unsqueeze(0))  # [N, DK]
                    w_idx = w_idx.clamp(0, x_q.shape[3] - 1)
                    h_idx = h_pos.unsqueeze(1).expand(-1, DK)
                    act_vals = x_q[b_idx, ic, h_idx, w_idx]  # [N_spatial, DK]

                    for bit_idx in range(self.BA):
                        # Extract bit bit_idx from each activation
                        act_bits = (act_vals >> bit_idx) & 1  # [N, DK]
                        # Pack DK bits into address: act_val = bit0 | bit1<<1 | bit2<<2
                        act_val = torch.zeros(N_spatial, dtype=torch.int64,
                                              device=x.device)
                        for g in range(DK):
                            act_val |= act_bits[:, g] << g
                        # Full LUT address: {wg_sel, act_bits}
                        addr = (wg_sel << self.G) | act_val  # [N_spatial]

                        # LUT lookup + routing: routed_table[s, p, addr] -> [D_p, N]
                        # Paper §4: per-step MUX selects which array for each output.
                        mac_results = self.routed_table[s, :, addr].T  # [N, D_p]

                        # Map D_p outputs to D_o channels:
                        # p = oc * DK + kr, out_ch = o_blk*64 + oc
                        oc_idx = torch.arange(64, device=x.device)
                        p_idx = oc_idx * DK + kr      # [64]
                        out_ch = o_blk * 64 + oc_idx  # [64]
                        valid = (out_ch < self.D_o) & (p_idx < self.D_p)
                        vals = mac_results[:, p_idx[valid]]  # [N, valid_count]
                        psum[:, out_ch[valid]] += vals << bit_idx

            output[b_idx] = psum.reshape(oH, oW, self.D_o).permute(2, 0, 1)

        return output

    def forward_scaled(self, x):
        """TLMAC conv with scaling to match N2UQ output.

        The LUT stores signed integer weights w_int = qi - WEIGHT_OFFSET.
        The TLMAC computes psum = conv(a, w_int) where a = round(x*n_q).
        N2UQ computes y = conv(x, qw) where qw = sf*(qi/n - clip/2).
        Since WEIGHT_OFFSET/n ≈ clip/2, the bias is negligible:
          y ≈ sf/(n*n_q) * psum
        """
        psum = self.forward(x)
        module = self.module
        sf = module.get_sf()  # [D_o, 1, 1, 1]
        n = float(2 ** module.num_bits - 1) / float(module.clip_val)
        n_q = (2 ** module.num_bits - 1) / 2.0
        sf_flat = sf.squeeze()  # [D_o]
        return psum * sf_flat.view(1, -1, 1, 1) / (n * n_q)

print('TLMACConvEmulator defined (3-bit weights, 3-bit activations, bit-serial)')


### Instantiate TLMAC Emulators

Load hex files from `tlmac_output/` and create one emulator per 3x3 conv layer.


In [ ]:
TLMAC_DIR = 'tlmac_output'
has_tlmac = os.path.isdir(os.path.join(TLMAC_DIR, 'weights'))

tlmac_emulators = {}  # layer_name -> TLMACConvEmulator
tlmac_meta = {}

if has_tlmac:
    with open(os.path.join(TLMAC_DIR, 'metadata.json')) as f:
        tlmac_meta = json.load(f)
    print(f'TLMAC output found: {len(tlmac_meta)} layers')
    for k, v in sorted(tlmac_meta.items()):
        print(f"  {k}: D_s={v['D_s']}, D_p={v['D_p']}, "
              f"{v['n_arr']} arrays, {v['clusters']} clusters, "
              f"{v['best_routes']} routes")

    for name, module in model_n2uq.named_modules():
        if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
            safe = name.replace('.', '_')
            if safe in tlmac_meta:
                layer_dir = os.path.join(TLMAC_DIR, 'weights', safe)
                tlmac_emulators[name] = TLMACConvEmulator(
                    layer_dir, module, tlmac_meta[safe]
                )
    print(f'\nInstantiated {len(tlmac_emulators)} TLMAC emulators')
else:
    print(f'TLMAC output not found at {TLMAC_DIR}/')
    print('Run tlmac_compile.ipynb first to generate hex files.')


## 3. LUT INIT Integrity Check

Verify that the LUT INIT hex files decode correctly: the bit-packing
and 2's-complement sign extension are self-consistent. This checks the
**emulator's hex-loading** is correct, not that TLMAC weights match N2UQ
weights (they don't — TLMAC weights go through clustering + routing SA).

For each LUT array, we decode the stored weight groups from the INIT
values and verify that `lut_table[ai, addr]` equals
$\sum_{g=0}^{G-1} a_g \cdot w_g$ for the decoded weights $w_g$.

The real accuracy comparison is in Section 4 (per-layer conv output),
which is apples-to-apples: same activations in, compare outputs out.


In [ ]:
n2uq_conv_layers = []
for name, module in model_n2uq.named_modules():
    if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
        n2uq_conv_layers.append((name, module))

print(f'LUT INIT integrity check: {len(n2uq_conv_layers)} layers')
print()

all_pass = True
for name, module in n2uq_conv_layers:
    if name not in tlmac_emulators:
        print(f'  {name}: no emulator, skipping')
        continue
    emu = tlmac_emulators[name]
    D_p = emu.D_p
    G = emu.G
    NCLUS = emu.NCLUS

    n_err = 0
    n_ok = 0

    # For each array, decode the weight groups stored in the LUT INIT
    # by probing specific activation patterns.
    # addr = {wg_sel[G_sel_bits-1:0], act_bits[G-1:0]}
    # For select sel and act pattern with only bit g set: addr = (sel << G) | (1 << g)
    # lut_table[ai, addr] should equal w_g (the g-th weight of that group).
    # For addr with all bits 0: lut_table should be 0 (all-zero activation).
    # For addr with all G bits set: lut_table should be sum of all w_g.
    for ai in range(min(emu.n_arr, 50)):  # sample up to 50 arrays
        for sel in range(NCLUS):
            # Zero activation: LUT should output 0
            addr_zero = (sel << G)
            zero_out = int(emu.lut_table[ai, addr_zero].item())
            if zero_out != 0:
                n_err += 1
            n_ok += 1

            # Single-bit activations: extract individual weights
            weights = []
            for g in range(G):
                addr_g = (sel << G) | (1 << g)
                w_g = int(emu.lut_table[ai, addr_g].item())
                weights.append(w_g)

            # All-bits-set: should equal sum of individual weights
            addr_all = (sel << G) | ((1 << G) - 1)
            expected_sum = sum(weights)
            actual_sum = int(emu.lut_table[ai, addr_all].item())
            if expected_sum != actual_sum:
                n_err += 1
            n_ok += 1

            # Verify linearity: bit0+bit1 should equal addr with bits 0,1 set
            for g0 in range(G):
                for g1 in range(g0 + 1, G):
                    addr_pair = (sel << G) | (1 << g0) | (1 << g1)
                    expected_pair = weights[g0] + weights[g1]
                    actual_pair = int(emu.lut_table[ai, addr_pair].item())
                    if expected_pair != actual_pair:
                        n_err += 1
                    n_ok += 1

    status = 'PASS' if n_err == 0 else f'FAIL ({n_err} mismatches)'
    if n_err > 0:
        all_pass = False
    print(f'  {name}: {status} ({n_ok} checks, {min(emu.n_arr, 50)} arrays sampled)')

print()
print(f'Overall: {"ALL PASS — LUT INIT decodes correctly" if all_pass else "FAILURES DETECTED — check hex loading"}')


## 4. Per-Layer Conv Comparison (TLMAC vs N2UQ) — Primary Accuracy Test

**This is the key accuracy test.** TLMAC weights differ from N2UQ weights
due to the clustering and routing SA steps in `tlmac_compile.ipynb`, so
we compare **conv outputs**, not weights. For each 3x3 conv layer, we
feed the **same** LTQ-quantized activations to both:
- **TLMAC**: bit-serial LUT lookup emulator (with scaling)
- **N2UQ**: `F.conv2d(x, qw)` with original quantized weights

This is apples-to-apples: same activations in, compare outputs out.
High correlation means the TLMAC architecture faithfully reproduces
the N2UQ convolution despite weight reorganization.


In [ ]:
def collect_ltq_activations(model, x_input):
    """Run N2UQ model with forward pre-hooks to capture the LTQ output
    (input) of each 3x3 conv layer.

    Returns: {layer_name: [B, D_i, H, W] tensor}
    """
    captured = {}
    pre_hooks = []

    def make_prehook(name):
        def prehook_fn(module, inp):
            captured[name] = inp[0].detach()
        return prehook_fn

    for name, module in model.named_modules():
        if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
            if name in tlmac_emulators:
                h = module.register_forward_pre_hook(make_prehook(name))
                pre_hooks.append(h)

    with torch.no_grad():
        model(x_input)

    for h in pre_hooks:
        h.remove()

    return captured


In [ ]:
# Capture LTQ activations from real test images
x_capture = x_test_images

print(f'Capturing LTQ activations from {N_IMG} test images...')
ltq_acts = collect_ltq_activations(model_n2uq, x_capture)
print(f'Captured {len(ltq_acts)} layer activations')

# Per-layer comparison
print()
print(f'{"Layer":<25s} {"Shape":>20s} {"Pearson r":>10s} {"Rel err":>10s} {"Max err":>10s}')
print('-' * 80)

layer_results = []
for name, module in n2uq_conv_layers:
    if name not in tlmac_emulators or name not in ltq_acts:
        continue
    emu = tlmac_emulators[name]
    x_ltq = ltq_acts[name]  # [B, D_i, H, W]

    # N2UQ reference: F.conv2d(x, qw)
    with torch.no_grad():
        qw = module.quantise_fwd()
        out_n2uq = F.conv2d(x_ltq, qw, stride=emu.stride, padding=emu.padding)

    # TLMAC emulator (with scaling)
    with torch.no_grad():
        out_tlmac = emu.forward_scaled(x_ltq)

    # Compare
    a = out_n2uq.flatten().float()
    b = out_tlmac.flatten().float()

    # Pearson correlation
    if a.std() > 1e-8 and b.std() > 1e-8:
        r = torch.corrcoef(torch.stack([a, b]))[0, 1].item()
    else:
        r = 0.0

    # Relative L2 error
    l2_err = (a - b).norm().item()
    l2_ref = a.norm().item()
    rel_err = l2_err / max(l2_ref, 1e-8)

    max_err = (a - b).abs().max().item()

    shape_str = f'{tuple(out_n2uq.shape)}'
    print(f'{name:<25s} {shape_str:>20s} {r:>10.4f} {rel_err:>10.4f} {max_err:>10.2f}')
    layer_results.append((name, r, rel_err, max_err))

print()
if layer_results:
    mean_r = sum(r[1] for r in layer_results) / len(layer_results)
    print(f'Mean Pearson r across {len(layer_results)} layers: {mean_r:.4f}')
    if mean_r > 0.99:
        print('TLMAC conv outputs closely match N2UQ (r > 0.99).')
    elif mean_r > 0.95:
        print('TLMAC conv outputs match N2UQ well (r > 0.95).')
    else:
        print('WARNING: TLMAC conv outputs diverge from N2UQ.')


### LUT Weight Verification

For each layer, verify that the LUT at the routed array stores the
correct N2UQ signed integer weight. This checks the SA placement,
route_map, and LUT content are mutually consistent.


In [ ]:
# Verify LUT weights match N2UQ weights via routing
print(f'{"Layer":<20s} {"Match%":>8s} {"Checked":>8s} {"Mismatch":>8s}')
print('-' * 50)

all_match = True
for name, module in n2uq_conv_layers:
    if name not in tlmac_emulators:
        continue
    emu = tlmac_emulators[name]
    D_o, D_i, Dk = emu.D_o, emu.D_i, emu.DK
    o_blocks = D_o // 64

    qi = module.get_integer_weights()
    w_int = qi.to(torch.int32) - emu.WEIGHT_OFFSET
    w_tensor = w_int.permute(1, 0, 2, 3).reshape(D_i, o_blocks, 64, Dk, Dk)
    w_tensor = w_tensor.reshape(emu.D_s, emu.D_p, Dk)

    n_ok = 0; n_err = 0
    for s in range(emu.D_s):
        ci = emu.cluster_map[s]
        for p in range(0, emu.D_p, 10):  # sample every 10th output
            ai = int(emu.route_map[s, p].item())
            for g in range(Dk):
                addr = (ci << emu.G) | (1 << g)
                lut_w = int(emu.lut_table[ai, addr].item())
                n2uq_w = int(w_tensor[s, p, g].item())
                if lut_w == n2uq_w:
                    n_ok += 1
                else:
                    n_err += 1
    total = n_ok + n_err
    pct = n_ok / total * 100 if total > 0 else 0
    status = 'OK' if n_err == 0 else 'MISMATCH'
    if n_err > 0:
        all_match = False
    print(f'{name:<20s} {pct:>7.1f}% {total:>8d} {n_err:>8d}  {status}')

print()
if all_match:
    print('ALL MATCH: LUT weights match N2UQ via routing.')
else:
    print('MISMATCHES FOUND: SA placement or route_map is inconsistent.')
    print('The LUT at the routed array does not store the expected weight.')


## 5. Full TLMAC Forward Pass

Run the entire ResNet-18 through TLMAC compute units using the hex files.
Each 3x3 conv layer uses bit-serial LUT lookup; all other layers
(LTQ, bias, PReLU, BN, skip connections, 1x1 downsample convs) use
the N2UQ model's parameters directly.

This is the end-to-end accuracy test: TLMAC vs N2UQ vs baseline.


In [ ]:
def tlmac_basic_block(block, x, emulators, block_name):
    """Forward through one BasicBlockQ using TLMAC for 3x3 conv layers.
    Non-conv layers use N2UQ parameters. Downsample 1x1 convs use N2UQ
    (TLMAC only targets 3x3 convs).
    """
    identity = x
    # bias11 -> prelu1 -> bias12 -> LTQ -> conv1 -> bn1
    out = block.bias12(block.prelu1(block.bias11(x)))
    out = block.quan1(out)
    conv1_name = block_name + ".conv1"
    if conv1_name in emulators:
        out = emulators[conv1_name].forward_scaled(out)
    else:
        out = block.conv1(out)
    out = block.bn1(out)

    # bias21 -> prelu2 -> bias22 -> LTQ -> conv2 -> bn2
    out = block.bias22(block.prelu2(block.bias21(out)))
    out = block.quan2(out)
    conv2_name = block_name + ".conv2"
    if conv2_name in emulators:
        out = emulators[conv2_name].forward_scaled(out)
    else:
        out = block.conv2(out)
    out = block.bn2(out)

    # Skip connection (downsample uses N2UQ — 1x1 conv not in TLMAC)
    if block.downsample is not None:
        identity = block.downsample(x)
    out += identity
    out = block.bias32(block.prelu3(block.bias31(out)))
    return out

def tlmac_resnet18(model, x, emulators):
    """Full ResNet-18 forward using TLMAC for all 3x3 conv layers."""
    x = model.relu(model.bn1(model.conv1(x)))
    x = model.maxpool(x)
    for bi, block in enumerate(model.layer1):
        x = tlmac_basic_block(block, x, emulators, "layer1." + str(bi))
    for bi, block in enumerate(model.layer2):
        x = tlmac_basic_block(block, x, emulators, "layer2." + str(bi))
    for bi, block in enumerate(model.layer3):
        x = tlmac_basic_block(block, x, emulators, "layer3." + str(bi))
    for bi, block in enumerate(model.layer4):
        x = tlmac_basic_block(block, x, emulators, "layer4." + str(bi))
    x = torch.flatten(model.avgpool(x), 1)
    return model.fc(x)

print('tlmac_resnet18 defined')


In [ ]:
# --- Run full-model comparison on real test images ---
# TLMAC bit-serial sim is slow, so we use the test images
x_eval = x_test_images
N_FULL = x_eval.shape[0]

print(f'Running {N_FULL} test images through 3 models...')

with torch.no_grad():
    logits_bl = model_bl(x_eval)
print('  Baseline done')

with torch.no_grad():
    logits_n2uq = model_n2uq(x_eval)
print('  N2UQ done')

if has_tlmac and tlmac_emulators:
    with torch.no_grad():
        logits_tlmac = tlmac_resnet18(model_n2uq, x_eval, tlmac_emulators)
    print('  TLMAC done')
else:
    logits_tlmac = logits_n2uq.clone()
    print('  TLMAC skipped (no hex files)')

# --- Top-1 agreement ---
top1_bl = logits_bl.argmax(1)
top1_n2uq = logits_n2uq.argmax(1)
top1_tlmac = logits_tlmac.argmax(1)

agree_n2uq_bl = (top1_n2uq == top1_bl).float().mean().item()
agree_tlmac_bl = (top1_tlmac == top1_bl).float().mean().item()
agree_tlmac_n2uq = (top1_tlmac == top1_n2uq).float().mean().item()

print(f'\n{"="*50}')
print(f'Samples: {N_FULL}')
print(f'Top-1 agreement N2UQ  vs BL:    {agree_n2uq_bl:.1%}')
print(f'Top-1 agreement TLMAC vs BL:    {agree_tlmac_bl:.1%}')
print(f'Top-1 agreement TLMAC vs N2UQ:  {agree_tlmac_n2uq:.1%}')

# --- L2 distances ---
dist_n2uq_bl = (logits_n2uq - logits_bl).norm(dim=1).mean().item()
dist_tlmac_bl = (logits_tlmac - logits_bl).norm(dim=1).mean().item()
dist_tlmac_n2uq = (logits_tlmac - logits_n2uq).norm(dim=1).mean().item()

print(f'\nMean L2 logit distance:')
print(f'  N2UQ  vs BL:   {dist_n2uq_bl:.2f}')
print(f'  TLMAC vs BL:   {dist_tlmac_bl:.2f}')
print(f'  TLMAC vs N2UQ: {dist_tlmac_n2uq:.2f}')

# --- Pearson correlation ---
corr_n2uq = torch.corrcoef(torch.stack([
    logits_bl.flatten(), logits_n2uq.flatten()]))[0, 1].item()
corr_tlmac = torch.corrcoef(torch.stack([
    logits_bl.flatten(), logits_tlmac.flatten()]))[0, 1].item()
corr_tlmac_n2uq = torch.corrcoef(torch.stack([
    logits_n2uq.flatten(), logits_tlmac.flatten()]))[0, 1].item()

print(f'\nLogit Pearson r:')
print(f'  N2UQ  vs BL:    {corr_n2uq:.4f}')
print(f'  TLMAC vs BL:    {corr_tlmac:.4f}')
print(f'  TLMAC vs N2UQ:  {corr_tlmac_n2uq:.4f}')


## 6. Visualization

Compare logit distributions and per-sample agreement across all three models.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-1 confidence
conf_bl = logits_bl.softmax(1).max(1).values.cpu().numpy()
conf_n2uq = logits_n2uq.softmax(1).max(1).values.cpu().numpy()
conf_tlmac = logits_tlmac.softmax(1).max(1).values.cpu().numpy()

axes[0, 0].hist(conf_bl, bins=10, alpha=0.4, label='Baseline')
axes[0, 0].hist(conf_n2uq, bins=10, alpha=0.4, label='N2UQ')
if has_tlmac and tlmac_emulators:
    axes[0, 0].hist(conf_tlmac, bins=10, alpha=0.4, label='TLMAC')
axes[0, 0].set_title('Top-1 Confidence')
axes[0, 0].legend()

# Per-sample L2 distance
dist_n2uq = (logits_n2uq - logits_bl).norm(dim=1).cpu().numpy()
dist_tlmac = (logits_tlmac - logits_bl).norm(dim=1).cpu().numpy()
x_pos = np.arange(N_FULL)
w = 0.35
axes[0, 1].bar(x_pos - w/2, dist_n2uq, w, label='N2UQ vs BL')
if has_tlmac and tlmac_emulators:
    axes[0, 1].bar(x_pos + w/2, dist_tlmac, w, label='TLMAC vs BL')
axes[0, 1].set_xlabel('Sample')
axes[0, 1].set_ylabel('L2 distance')
axes[0, 1].set_title('Per-sample logit distance from baseline')
axes[0, 1].legend()

# Max logit scatter: N2UQ vs Baseline
axes[1, 0].scatter(logits_bl.max(1).values.cpu().numpy(),
                   logits_n2uq.max(1).values.cpu().numpy(),
                   alpha=0.6, s=30, label='N2UQ')
if has_tlmac and tlmac_emulators:
    axes[1, 0].scatter(logits_bl.max(1).values.cpu().numpy(),
                       logits_tlmac.max(1).values.cpu().numpy(),
                       alpha=0.6, s=30, label='TLMAC', marker='x')
mn = min(logits_bl.max(1).values.min().item(),
         logits_n2uq.max(1).values.min().item())
mx = max(logits_bl.max(1).values.max().item(),
         logits_n2uq.max(1).values.max().item())
axes[1, 0].plot([mn, mx], [mn, mx], 'r--', alpha=0.5)
axes[1, 0].set_xlabel('Baseline max logit')
axes[1, 0].set_ylabel('Quantised max logit')
axes[1, 0].set_title('Max logit comparison')
axes[1, 0].legend()

# TLMAC vs N2UQ max logit
if has_tlmac and tlmac_emulators:
    axes[1, 1].scatter(logits_n2uq.max(1).values.cpu().numpy(),
                        logits_tlmac.max(1).values.cpu().numpy(),
                        alpha=0.6, s=30)
    mn = min(logits_n2uq.max(1).values.min().item(),
             logits_tlmac.max(1).values.min().item())
    mx = max(logits_n2uq.max(1).values.max().item(),
             logits_tlmac.max(1).values.max().item())
    axes[1, 1].plot([mn, mx], [mn, mx], 'r--', alpha=0.5)
    axes[1, 1].set_xlabel('N2UQ max logit')
    axes[1, 1].set_ylabel('TLMAC max logit')
    axes[1, 1].set_title('TLMAC vs N2UQ max logit')
else:
    axes[1, 1].text(0.5, 0.5, 'No TLMAC data',
                     ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()


## 7. Summary


In [ ]:
print('=' * 65)
print('  Model Comparison Summary')
print('=' * 65)
print()
hdr = f'{"Model":<22s} {"vs BL top1":>11s} {"vs BL L2":>9s} {"vs BL r":>8s}'
print(hdr)
print('-' * 55)
print(f'{"Baseline":<22s} {"---":>11s} {"---":>9s} {"---":>8s}')
print(f'{"N2UQ (3-bit)":<22s} {agree_n2uq_bl:>11.1%} {dist_n2uq_bl:>9.2f} {corr_n2uq:>8.4f}')
if has_tlmac and tlmac_emulators:
    print(f'{"TLMAC (LUT sim)":<22s} {agree_tlmac_bl:>11.1%} {dist_tlmac_bl:>9.2f} {corr_tlmac:>8.4f}')
    print()
    total_arrs = sum(v['n_arr'] for v in tlmac_meta.values())
    total_lut6 = total_arrs * tlmac_meta[list(tlmac_meta.keys())[0]]['NLUT_PER_ARR']
    print(f'TLMAC: {len(tlmac_emulators)} layers, {total_arrs} LUT arrays, '
          f'{total_lut6} LUT-6 total')
    print()
    print(f'TLMAC vs N2UQ:  top-1 agree={agree_tlmac_n2uq:.1%}, '
          f'L2={dist_tlmac_n2uq:.2f}, r={corr_tlmac_n2uq:.4f}')
    print()
    if 'all_pass' in dir() and all_pass:
        print('LUT INIT integrity check: ALL PASS')
    if agree_tlmac_n2uq >= 0.9 * max(agree_n2uq_bl, 0.01):
        print('Result: TLMAC accuracy is comparable to N2UQ.')
    else:
        print('Result: WARNING — TLMAC accuracy diverges from N2UQ.')
        print('  Check weight extraction and activation quantization.')
print()
print('=' * 65)
